# Inner-Boundary Magnetic Field Map (ZDI-Style Starter)

This example loads the 3D BATSRUS sample, samples the **inner boundary shell** (`R=1`) using the library, and makes lat/lon magnetic maps.

The notebook is intentionally thin: it uses library helpers for shell sampling and plotting instead of reimplementing the workflow in cells.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from starwinds_analysis.smart_ds import SmartDs
from starwinds_analysis.data.samples import get_sample
from starwinds_analysis.analysis.shells import infer_body_radius_m, integrate_shell_scalar
from starwinds_analysis.analysis.shell_magnetic import (
    plot_magnetic_zdi_triplet,
    plot_shell_tangential_vectors_lonlat,
    sample_shell_magnetic_field_map,
)

plt.rcParams['figure.dpi'] = 120


## Load the 3D Sample and Configure the Radius Scale

We feed the stellar radius into the BATSRUS/griblet graph near the top so SI conversions are available consistently.

In [ ]:
DATA_FILE = Path(get_sample('3d__var_3_n00060000.plt'))
STAR_RADIUS_M = 696_000_000.0

print('Using:', DATA_FILE)
sds = SmartDs.from_file(str(DATA_FILE))
sds.add_batsrus_graph(body_radius_m=STAR_RADIUS_M)
body_radius_m = infer_body_radius_m(sds)

print('Title:', sds.title)
print('Zone :', sds.zone)
print('Points:', len(sds.points))
print('Variables:', len(sds.variables))
print(f'Body radius [m]: {body_radius_m:.6e}')


## Sample the Inner Boundary Shell (`R = 1`)

This returns radial/azimuthal/meridional/tangential magnetic components on a regular spherical sampling grid.

- Internal field values are SI (`T`)
- Plotting unit is chosen separately (below we use `G`)

In [ ]:
R_BOUNDARY = 1.0
N_POLAR = 64
N_AZIMUTH = 128
B_PLOT_UNIT = "G"

shell_b = sample_shell_magnetic_field_map(
    sds,
    R_BOUNDARY,
    n_polar=N_POLAR,
    n_azimuth=N_AZIMUTH,
    method="nearest",
    length_unit_to_m=body_radius_m,
)

print('Shell grid shape (theta, phi):', shell_b.theta.shape)
print('Radius:', shell_b.radius)
print('Finite B_r cells:', int(np.isfinite(shell_b.b_r_T).sum()), '/', shell_b.b_r_T.size)


## ZDI-Style Component Maps (Radial / Azimuthal / Meridional)

Conventions used here:

- **Radial**: outward normal component (`B_r`)
- **Azimuthal**: `+phi` direction (eastward on the lat/lon map)
- **Meridional**: northward tangent component (`-B_theta`) so it follows latitude intuition

In [ ]:
fig, axes = plot_magnetic_zdi_triplet(shell_b, unit=B_PLOT_UNIT, figsize=(10, 10))
plt.show()


## Tangential Field as Vectors on a Lat/Lon Map

This plot shows the tangential field in two ways:

- **Background color**: tangential magnetic strength `|B_tan|`
- **Arrows**: tangential field direction in the local surface tangent plane

Arrow meaning (important):

- Arrow **direction** follows the tangential field direction (`B_phi`, meridional)
- Arrow **length is normalized** (fixed plotting length), so it does **not** encode field magnitude
- Tangential field magnitude is encoded by the background colors
- The east-west arrow component is corrected by `cos(latitude)` so the direction is sensible on a lon/lat map

In [ ]:
fig, ax, extra = plot_shell_tangential_vectors_lonlat(
    shell_b,
    unit=B_PLOT_UNIT,
    background="tangential",
    background_scale="positive_log",
    arrow_stride=(4, 6),
    normalize_arrows=True,
    arrow_length_deg=8.0,
)

# Optional but useful: overlay the radial-field polarity inversion line (B_r = 0).
b_r_plot = shell_b.component("radial", unit=B_PLOT_UNIT)
ax.contour(shell_b.lon_deg, shell_b.lat_deg, b_r_plot, levels=[0.0], colors="black", linewidths=0.8, alpha=0.7)
ax.set_title(f"Tangential field directions at R={shell_b.radius:g} (background |B_tan|)")

if "n_nonpositive" in extra:
    print("Non-positive |B_tan| cells shown with under-color:", extra["n_nonpositive"])

plt.show()


## Quick Boundary Summary (Useful Extra)

A few easy quantities to compute from the same shell sample:

- signed radial flux (should be small for a closed spherical surface in a divergence-free field)
- unsigned radial flux
- RMS strengths of radial / azimuthal / meridional / tangential components

In [ ]:
def _rms(values):
    arr = np.asarray(values, dtype=float)
    finite = arr[np.isfinite(arr)]
    return float(np.sqrt(np.mean(finite * finite))) if finite.size else np.nan

signed_flux_Wb, signed_cov = integrate_shell_scalar(shell_b.b_r_T[None, ...], shell_b.shell_samples.area[:1])
unsigned_flux_Wb, unsigned_cov = integrate_shell_scalar(np.abs(shell_b.b_r_T)[None, ...], shell_b.shell_samples.area[:1])

print(f"Signed radial flux   [Wb]: {float(signed_flux_Wb[0]):.6e}  (coverage={float(signed_cov[0]):.3f})")
print(f"Unsigned radial flux [Wb]: {float(unsigned_flux_Wb[0]):.6e}  (coverage={float(unsigned_cov[0]):.3f})")
print()
print(f"RMS B_r          [{B_PLOT_UNIT}]: {_rms(shell_b.component('radial', unit=B_PLOT_UNIT)):.6e}")
print(f"RMS B_azimuthal  [{B_PLOT_UNIT}]: {_rms(shell_b.component('azimuthal', unit=B_PLOT_UNIT)):.6e}")
print(f"RMS B_meridional [{B_PLOT_UNIT}]: {_rms(shell_b.component('meridional', unit=B_PLOT_UNIT)):.6e}")
print(f"RMS |B_tan|      [{B_PLOT_UNIT}]: {_rms(shell_b.component('tangential', unit=B_PLOT_UNIT)):.6e}")


## Easy Follow-Ups

- compare two timesteps side-by-side using the same plotting helpers
- switch `background="radial"` in the vector plot to show tangential arrows over radial polarity
- repeat at `R > 1` to see how the magnetic topology opens with height